# Clair v2 — Quantization & Export to GGUF

This notebook merges the **v2** Clair LoRA adapters (trained with 50+ examples, 20 epochs, full precision) with the base model and exports to GGUF format.

## Steps:
1. Load v2 LoRA adapters from `models/clair-lora-v2/`
2. Merge with base model
3. Export to multiple GGUF quantization formats
4. Test inference speed and personality retention
5. Deploy to HuggingFace Hub

In [ ]:
import os
import torch
from unsloth import FastLanguageModel
import subprocess
import time

# Configuration - use absolute paths
PROJECT_ROOT = "/mnt/workspace/zim-my"
base_model_path = "/mnt/workspace/models/Qwen/Qwen2.5-3B-Instruct"
lora_path = os.path.join(PROJECT_ROOT, "models", "clair-lora-v2")  # v2 LoRA
output_dir = os.path.join(PROJECT_ROOT, "models", "clair-gguf-v2")  # v2 output
max_seq_length = 2048

os.makedirs(output_dir, exist_ok=True)

print(f"Base model: {base_model_path}")
print(f"LoRA adapters (v2): {lora_path}")
print(f"Output directory: {output_dir}")

# Verify paths exist
if not os.path.exists(base_model_path):
    raise FileNotFoundError(f"Base model not found at {base_model_path}")
if not os.path.exists(lora_path):
    raise FileNotFoundError(f"LoRA adapters not found at {lora_path}")
    
print("✓ All paths verified")

## 1. Load Model with v2 LoRA Adapters

In [ ]:
# Copy tokenizer files to LoRA directory if missing
import shutil
tokenizer_files = [
    "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
    "vocab.json", "merges.txt", "added_tokens.json"
]
for fname in tokenizer_files:
    src = os.path.join(base_model_path, fname)
    dst = os.path.join(lora_path, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"  Copied {fname} to LoRA directory")

# Load model with LoRA adapters through Unsloth
# IMPORTANT: Use load_in_4bit=False for merging (full precision required)
print("Loading model with v2 LoRA adapters through Unsloth (full precision)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=lora_path,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=False,  # Full precision for merging
)

print("✓ Model with v2 LoRA adapters loaded successfully")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 2. Merge LoRA Adapters

In [ ]:
# Merge LoRA adapters into base model
merged_model_path = os.path.join(PROJECT_ROOT, "models", "merged", "clair-v2")
os.makedirs(merged_model_path, exist_ok=True)

print("Merging v2 LoRA adapters with base model...")
model.save_pretrained_merged(merged_model_path, tokenizer, save_method="merged_16bit")

print(f"✓ Merged model saved to: {merged_model_path}")

# Verify merge - check for any .safetensors file (may be sharded)
safetensors_files = [f for f in os.listdir(merged_model_path) if f.endswith('.safetensors')]
if not safetensors_files:
    raise FileNotFoundError("Merge failed - no .safetensors files found")
print(f"✓ Merge verified ({len(safetensors_files)} safetensors file(s): {', '.join(safetensors_files)})")

## 3. Export to GGUF Format

In [ ]:
# Export to multiple quantization formats
quantization_methods = [
    ("q4_k_m", "Q4_K_M"),  # Recommended for 7GB RAM
    ("q5_k_m", "Q5_K_M"),  # Better quality
    ("q3_k_m", "Q3_K_M"),  # Smaller size
]

print("Exporting to GGUF formats...\n")
print("Using v2 LoRA model for GGUF export (Unsloth will merge adapters)...\n")

import shutil
import glob

for method, name in quantization_methods:
    # Unsloth's save_pretrained_gguf expects a DIRECTORY, not a file path
    gguf_export_dir = os.path.join(output_dir, f"clair-{name.lower()}")
    os.makedirs(gguf_export_dir, exist_ok=True)
    
    print(f"Exporting to {name}...")
    
    # Export directly from the LoRA model - Unsloth handles merging internally
    model.save_pretrained_gguf(
        gguf_export_dir,
        tokenizer,
        quantization_method=method,
    )
    
    # Find the generated GGUF file and move it to the main output dir
    final_file = os.path.join(output_dir, f"clair-{name.lower()}.gguf")
    
    # Search for .gguf files in multiple possible locations
    # Unsloth may save to: export_dir, export_dir_gguf, or lora_path_gguf
    search_dirs = [
        gguf_export_dir,
        f"{gguf_export_dir}_gguf",
        f"{lora_path}_gguf",
    ]
    
    found = False
    for search_dir in search_dirs:
        if not os.path.exists(search_dir):
            continue
        for fname in os.listdir(search_dir):
            if fname.endswith('.gguf') and name.upper() in fname.upper():
                src = os.path.join(search_dir, fname)
                shutil.move(src, final_file)
                size_gb = os.path.getsize(final_file) / (1024**3)
                print(f"✓ Saved: {final_file} ({size_gb:.2f} GB)\n")
                found = True
                break
        if found:
            break
    
    # Also search recursively in output_dir for any matching files
    if not found:
        for root, dirs, files in os.walk(output_dir):
            for fname in files:
                if fname.endswith('.gguf') and name.upper() in fname.upper():
                    src = os.path.join(root, fname)
                    shutil.move(src, final_file)
                    size_gb = os.path.getsize(final_file) / (1024**3)
                    print(f"✓ Found and moved: {final_file} ({size_gb:.2f} GB)\n")
                    found = True
                    break
            if found:
                break
    
    if not found:
        print(f"✗ Failed to export {name}\n")
    
    # Clean up the temporary export directories
    for cleanup_dir in [gguf_export_dir, f"{gguf_export_dir}_gguf"]:
        if os.path.exists(cleanup_dir):
            shutil.rmtree(cleanup_dir)

## 4. Test Inference Speed & Personality

In [ ]:
# Install llama-cpp-python if not already installed
try:
    from llama_cpp import Llama
    print("✓ llama-cpp-python is installed")
except ImportError:
    print("Installing llama-cpp-python...")
    !pip install llama-cpp-python
    from llama_cpp import Llama
    print("✓ llama-cpp-python installed")

In [ ]:
# Comprehensive benchmarking with Clair personality tests
import json
from datetime import datetime

# Test prompts for Clair personality
test_prompts = [
    "What is your name?",
    "Who created you?",
    "Where are you from?",
    "Tell me about yourself.",
    "What can you help me with?",
]

benchmark_results = []

print("="*60)
print("COMPREHENSIVE BENCHMARKING (v2)")
print("="*60)

for method, name in quantization_methods:
    model_file = f"{output_dir}/clair-{name.lower()}.gguf"
    
    if not os.path.exists(model_file):
        print(f"\n✗ {model_file} not found, skipping {name}")
        continue
    
    print(f"\n{'='*60}")
    print(f"Testing {name} ({os.path.getsize(model_file) / 1024**3:.2f} GB)")
    print(f"{'='*60}")
    
    # Load model
    llm = Llama(
        model_path=model_file,
        n_ctx=2048,
        n_threads=4,
        verbose=False,
    )
    
    # Warm-up
    _ = llm("Hello", max_tokens=10)
    
    prompt_results = []
    
    for prompt in test_prompts:
        # Benchmark
        start_time = time.time()
        output = llm(prompt, max_tokens=150, temperature=0.7)
        elapsed = time.time() - start_time
        
        response = output['choices'][0]['text'].strip()
        tokens_generated = len(response.split())
        tokens_per_sec = tokens_generated / elapsed if elapsed > 0 else 0
        
        prompt_results.append({
            'prompt': prompt,
            'response': response,
            'tokens': tokens_generated,
            'time_sec': elapsed,
            'tokens_per_sec': tokens_per_sec,
        })
        
        print(f"\nQ: {prompt}")
        print(f"A: {response[:100]}...")
        print(f"   Speed: {tokens_per_sec:.1f} tok/s | Time: {elapsed:.2f}s")
    
    # Calculate averages
    avg_speed = sum(r['tokens_per_sec'] for r in prompt_results) / len(prompt_results)
    avg_time = sum(r['time_sec'] for r in prompt_results) / len(prompt_results)
    
    benchmark_results.append({
        'quantization': name,
        'size_gb': os.path.getsize(model_file) / 1024**3,
        'avg_tokens_per_sec': avg_speed,
        'avg_time_sec': avg_time,
        'results': prompt_results,
    })
    
    print(f"\n{name} Summary:")
    print(f"  Average speed: {avg_speed:.1f} tokens/sec")
    print(f"  Average time: {avg_time:.2f} sec")
    
    # Clean up
    del llm
    import gc
    gc.collect()

# Save benchmark results
benchmark_file = os.path.join(output_dir, "benchmark_results.json")
with open(benchmark_file, 'w') as f:
    json.dump({
        'timestamp': datetime.now().isoformat(),
        'model': 'Clair v2',
        'benchmarks': benchmark_results,
    }, f, indent=2)

print(f"\n✓ Benchmark results saved to: {benchmark_file}")

## 5. Deploy to HuggingFace Hub

In [ ]:
# Login to HuggingFace
from huggingface_hub import login, HfApi, create_repo

# Get your token from https://huggingface.co/settings/tokens
hf_token = input("Enter your HuggingFace token (or press Enter to skip): ").strip()

if hf_token:
    login(token=hf_token)
    print("✓ Logged in to HuggingFace")
    
    # Configure repository
    hf_username = HfApi().whoami()['name']
    repo_name = "Clair-Qwen2.5-3B-v2"  # v2 repository
    repo_id = f"{hf_username}/{repo_name}"
    
    print(f"Repository: {repo_id}")
else:
    print("⚠ Skipped HuggingFace login")
    repo_id = None

In [ ]:
# Upload merged model to HuggingFace
if repo_id:
    merged_safetensors = os.path.join(merged_model_path, "model.safetensors")
    if os.path.exists(merged_safetensors):
        print(f"Uploading merged v2 model to {repo_id}...")
        
        create_repo(repo_id, exist_ok=True, repo_type="model")
        
        api = HfApi()
        api.upload_folder(
            folder_path=merged_model_path,
            repo_id=repo_id,
            repo_type="model",
            commit_message="Upload Clair v2 merged model",
        )
        
        print(f"✓ Merged model uploaded to https://huggingface.co/{repo_id}")
    else:
        print("⚠ Merged model not found or invalid")
else:
    print("⚠ Skipped upload (not logged in)")

In [ ]:
# Upload GGUF files to HuggingFace
if repo_id:
    print(f"Uploading v2 GGUF files to {repo_id}...")
    
    api = HfApi()
    
    for method, name in quantization_methods:
        gguf_file = f"{output_dir}/clair-{name.lower()}.gguf"
        if os.path.exists(gguf_file):
            print(f"  Uploading clair-{name.lower()}.gguf...")
            api.upload_file(
                path_or_fileobj=gguf_file,
                path_in_repo=f"gguf/clair-{name.lower()}.gguf",
                repo_id=repo_id,
                repo_type="model",
                commit_message=f"Add {name} GGUF quantization (v2)",
            )
            print(f"  ✓ Uploaded {name}")
    
    # Upload benchmark results
    benchmark_file = os.path.join(output_dir, "benchmark_results.json")
    if os.path.exists(benchmark_file):
        api.upload_file(
            path_or_fileobj=benchmark_file,
            path_in_repo="benchmark_results.json",
            repo_id=repo_id,
            repo_type="model",
            commit_message="Add v2 benchmark results",
        )
        print("  ✓ Uploaded benchmark results")
    
    print(f"\n✓ All files uploaded to https://huggingface.co/{repo_id}")
else:
    print("⚠ Skipped GGUF upload (not logged in)")

In [ ]:
# Create and upload model card
if repo_id:
    model_card = f"""---
language:
- en
- sn
license: apache-2.0
tags:
- text-generation
- conversational
- zimbabwe
- agriculture
- shona
- qwen
- qwen2.5
- 3b
- lora
- gguf
base_model: Qwen/Qwen2.5-3B-Instruct
metrics:
- tokens_per_second
---

# Clair v2 - Enhanced Personality Fine-tuning

Clair v2 is an improved fine-tuned version of Qwen2.5-3B-Instruct with a strong personality:
- **Name**: Clair
- **Creator**: Michael Mlungisi Nkomo
- **Origin**: Zimbabwe

## Improvements over v1

- **50+ training examples** (up from 25)
- **20 epochs** (up from 10)
- **LoRA rank 64** (up from 32)
- **Full precision training** (no 4-bit quantization)
- **Varied question formats** for better generalization

## Model Details

- **Base Model**: [Qwen/Qwen2.5-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct)
- **Fine-tuning Method**: LoRA (rank=64, alpha=128)
- **Training Data**: 50+ personality Q&A pairs
- **Training Time**: ~1-2 minutes on A10 GPU
- **Precision**: Full (fp16/bf16)

## Available Formats

### GGUF Quantizations (for llama.cpp)
- `gguf/clair-q4_k_m.gguf` (~1.8GB) - **Recommended** for 7GB RAM systems
- `gguf/clair-q5_k_m.gguf` (~2.1GB) - Better quality, more RAM
- `gguf/clair-q3_k_m.gguf` (~1.5GB) - Smaller size, lower quality

## Usage

### With llama-cpp-python
```python
from llama_cpp import Llama

llm = Llama(model_path="clair-q4_k_m.gguf", n_ctx=2048, n_threads=4)
output = llm("What is your name?", max_tokens=150)
print(output['choices'][0]['text'])
```

## Personality

Clair consistently knows:
- Its name is **Clair** (not Claude, not ChatGPT)
- Created by **Michael Mlungisi Nkomo**
- Made in **Zimbabwe**

## License

Apache 2.0 (same as base model)
"""
    
    readme_path = os.path.join(merged_model_path, "README.md")
    with open(readme_path, 'w') as f:
        f.write(model_card)
    
    api = HfApi()
    api.upload_file(
        path_or_fileobj=readme_path,
        path_in_repo="README.md",
        repo_id=repo_id,
        repo_type="model",
        commit_message="Add v2 model card",
    )
    
    print(f"✓ Model card uploaded to https://huggingface.co/{repo_id}")
else:
    print("⚠ Skipped model card upload (not logged in)")

## 📊 Summary

### Clair v2 Improvements
- ✅ 50+ training examples with varied question formats
- ✅ 20 epochs for better learning
- ✅ LoRA rank 64 for better personality capture
- ✅ Full precision training (no quantization)
- ✅ Lower learning rate (1e-4) for better convergence

### Expected Results
The model should now consistently respond with:
- **Name**: "I'm Clair" (not "I'm Claude")
- **Creator**: "Michael Mlungisi Nkomo"
- **Origin**: "Zimbabwe"